# Parte individual: grafo de colaboración de Harrison Ford

**Análisis y Ciencia de Datos — Proyecto final**
Herman Echeverría

El notebook corre de principio a fin **sin API key**, leyendo solo de `cache/`.
La key solo se usa para llenar el cache la primera vez: se lee de la variable de entorno
`TMDB_API_KEY` y nunca se escribe en el notebook.

In [ ]:
import os
import re
import json
import time
import itertools
from collections import deque, Counter, defaultdict
from pathlib import Path

import numpy as np
import requests
import matplotlib.pyplot as plt
import networkx as nx            # solo para visualizar y verificar
from sklearn.cluster import KMeans

In [ ]:
# Configuración
CACHE_DIR = Path("cache")
BASE_URL = "https://api.themoviedb.org/3"

SEED_ID = 3                      # Harrison Ford (verificado con GET /person/3)
SEED_NAME = "Harrison Ford"
LIMIT = 5
START_DATE = "1975-01-01"
END_DATE = "2025-12-31"

API_KEY = os.environ.get("TMDB_API_KEY")   # None si no hay key: entonces solo se lee el cache

## I.2 Clase `Graph`

Grafo **no dirigido y con pesos**. Cada arista se guarda con la llave `(min(a,b), max(a,b))`,
así $(a,b)$ y $(b,a)$ son la misma arista.

In [ ]:
class Graph:
    def __init__(self):
        self.nodes = {}   # id -> name
        self.edges = {}   # (id menor, id mayor) -> peso

    @staticmethod
    def _key(a, b):
        return (a, b) if a <= b else (b, a)

    def add_node(self, node):
        """node es una tupla (id, name). La agrega solo si no existe."""
        node_id, name = node
        if node_id not in self.nodes:
            self.nodes[node_id] = name

    def add_edge(self, a, b):
        """Agrega la arista entre los id a y b con peso 1, o le suma 1 si ya existe."""
        if a == b:
            return
        key = self._key(a, b)
        self.edges[key] = self.edges.get(key, 0) + 1

    def total_nodes(self):
        return len(self.nodes)

    def total_edges(self):
        return len(self.edges)

    def degree(self, node_id):
        """Número de aristas que tocan al nodo (no la suma de pesos)."""
        return sum(1 for a, b in self.edges if node_id in (a, b))

    def degrees(self):
        deg = {n: 0 for n in self.nodes}
        for a, b in self.edges:
            deg[a] += 1
            deg[b] += 1
        return deg

    def max_degree_nodes(self):
        """Diccionario {id: grado} con el nodo (o los nodos, si hay empate) de grado máximo."""
        deg = self.degrees()
        if not deg:
            return {}
        m = max(deg.values())
        return {n: d for n, d in deg.items() if d == m}

    def adjacency(self):
        """Devuelve (A, ids): A es la matriz de adyacencia con pesos (numpy) e ids el orden de sus filas."""
        ids = list(self.nodes)
        pos = {n: i for i, n in enumerate(ids)}
        A = np.zeros((len(ids), len(ids)))
        for (a, b), w in self.edges.items():
            A[pos[a], pos[b]] = w
            A[pos[b], pos[a]] = w
        return A, ids

In [ ]:
# Pruebas pequeñas de la clase
g_test = Graph()
g_test.add_node((1, "A")); g_test.add_node((2, "B")); g_test.add_node((3, "C"))
g_test.add_node((1, "A repetido"))
g_test.add_edge(1, 2); g_test.add_edge(2, 1); g_test.add_edge(2, 3)
assert g_test.total_nodes() == 3 and g_test.nodes[1] == "A"
assert g_test.total_edges() == 2 and g_test.edges[(1, 2)] == 2
assert g_test.max_degree_nodes() == {2: 2}
A_test, ids_test = g_test.adjacency()
assert np.allclose(A_test, A_test.T) and A_test[0, 1] == 2
print("Graph OK")

## I.3 Clase `TMDBAPIUtils`

Cada respuesta del API se guarda como JSON en `cache/people/<person_id>.json` y
`cache/movies/<movie_id>.json`, y se lee de ahí si ya existe.

In [ ]:
SELF_ROLE = re.compile(r"\b(him|her|them)?self\b", re.IGNORECASE)


class TMDBAPIUtils:
    def __init__(self, api_key=None, cache_dir=CACHE_DIR):
        self.api_key = api_key
        self.cache_dir = Path(cache_dir)

    def _get(self, endpoint, cache_file):
        """Lee cache_file si existe; si no, llama al API (endpoint), guarda la respuesta y la devuelve."""
        cache_file = Path(cache_file)
        if cache_file.exists():
            return json.loads(cache_file.read_text(encoding="utf-8"))
        if not self.api_key:
            raise RuntimeError(f"{cache_file} no está en el cache y no hay TMDB_API_KEY")
        for intento in range(5):
            try:
                r = requests.get(BASE_URL + endpoint,
                                 params={"api_key": self.api_key, "language": "en-US"}, timeout=30)
            except requests.RequestException as e:
                # sin "from e": el mensaje original trae la URL con la API key
                raise RuntimeError(f"Fallo de red pidiendo {endpoint}: {type(e).__name__}") from None
            if r.status_code == 429:          # demasiadas solicitudes: esperar y reintentar
                time.sleep(2 ** intento)
                continue
            if not r.ok:
                raise RuntimeError(f"TMDB respondió {r.status_code} para {endpoint}")
            break
        data = r.json()
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        cache_file.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        return data

    def get_movie_cast(self, movie_id, limit=None, exclude_ids=None):
        """Elenco de la película: solo miembros con 0 <= order <= limit - 1 y cuyo id no esté en exclude_ids."""
        data = self._get(f"/movie/{movie_id}/credits", self.cache_dir / "movies" / f"{movie_id}.json")
        exclude = set(exclude_ids or [])
        cast = []
        for c in data.get("cast", []):
            if c["order"] < 0 or (limit is not None and c["order"] > limit - 1):
                continue
            if c["id"] in exclude:
                continue
            cast.append({"id": c["id"], "name": c["name"], "order": c["order"],
                         "character": c.get("character", "")})
        return cast

    def get_movie_credits_for_person(self, person_id, start_date=None, end_date=None):
        """Una entrada por película en la que actuó la persona: {id, title, release_date, order}.
        Filtra por fecha, quita documentales (género 99) y papeles donde se interpreta a sí misma."""
        data = self._get(f"/person/{person_id}/movie_credits", self.cache_dir / "people" / f"{person_id}.json")
        movies = {}
        for c in data.get("cast", []):
            rd = c.get("release_date") or ""
            if not rd:                                    # sin fecha no se puede saber si cae en la ventana
                continue
            if (start_date and rd < start_date) or (end_date and rd > end_date):
                continue
            if 99 in c.get("genre_ids", []):
                continue
            if SELF_ROLE.search(c.get("character") or ""):
                continue
            # una persona puede salir dos veces en la misma película: nos quedamos con su mejor order
            if c["id"] not in movies or c["order"] < movies[c["id"]]["order"]:
                movies[c["id"]] = {"id": c["id"], "title": c["title"],
                                   "release_date": rd, "order": c["order"]}
        return list(movies.values())

In [ ]:
tmdb = TMDBAPIUtils(api_key=API_KEY)

## I.4 Construcción del grafo

`limit = 5`, ventana del 1 de enero de 1975 al 31 de diciembre de 2025.
Una película **cuenta** para un actor si salió dentro de la ventana y el actor aparece con `order < 5`.

In [ ]:
def counted_movies(person_id):
    return [m for m in tmdb.get_movie_credits_for_person(person_id, START_DATE, END_DATE)
            if m["order"] < LIMIT]

G = Graph()

# (a) La semilla
G.add_node((SEED_ID, SEED_NAME))

# (b) Para cada película que cuenta para la semilla, los demás del elenco con order < 5
seed_movies = counted_movies(SEED_ID)
for m in seed_movies:
    for c in tmdb.get_movie_cast(m["id"], limit=LIMIT, exclude_ids=[SEED_ID]):
        G.add_node((c["id"], c["name"]))

# (c) Para cada nodo, sus créditos. movie_nodes[película] = nodos del grafo para los que cuenta.
#     Al usar un conjunto, cada pareja recibe una sola arista por película aunque la película
#     aparezca en los créditos de los dos.
movie_nodes = defaultdict(set)
movie_title = {}
for node_id in list(G.nodes):
    for m in counted_movies(node_id):
        movie_nodes[m["id"]].add(node_id)
        movie_title[m["id"]] = f'{m["title"]} ({m["release_date"][:4]})'

for movie_id, members in movie_nodes.items():
    for a, b in itertools.combinations(sorted(members), 2):
        G.add_edge(a, b)

print(f"Películas que cuentan para {SEED_NAME}: {len(seed_movies)}")

In [ ]:
print("Nodos:  ", G.total_nodes())
print("Aristas:", G.total_edges())
print("Nodo(s) de mayor grado:", {G.nodes[n]: d for n, d in G.max_degree_nodes().items()})

## I.5 Visualización

In [ ]:
A, ids = G.adjacency()
names = [G.nodes[n] for n in ids]
deg = G.degrees()
deg_arr = np.array([deg[n] for n in ids])

nxG = nx.Graph()
nxG.add_nodes_from(range(len(ids)))
for (a, b), w in G.edges.items():
    nxG.add_edge(ids.index(a), ids.index(b), weight=w)

pos = nx.spring_layout(nxG, seed=0, k=0.35, iterations=200)
fig, ax = plt.subplots(figsize=(14, 11))
widths = [0.3 + 0.6 * nxG[u][v]["weight"] for u, v in nxG.edges]
nx.draw_networkx_edges(nxG, pos, ax=ax, width=widths, alpha=0.25, edge_color="gray")
nodes = nx.draw_networkx_nodes(nxG, pos, ax=ax, node_size=20 + 8 * deg_arr,
                               node_color=deg_arr, cmap="viridis",
                               vmax=np.sort(deg_arr)[-2])   # sin esto la semilla aplasta la escala de color
top = np.argsort(-deg_arr)[:20]
nx.draw_networkx_labels(nxG, pos, labels={i: names[i] for i in top}, font_size=8, ax=ax)
fig.colorbar(nodes, ax=ax, shrink=0.6, label="grado")
ax.set_title(f"Grafo de colaboración de {SEED_NAME} (1975–2025, order < {LIMIT})")
ax.axis("off")
fig.tight_layout()
fig.savefig("grafo.png", dpi=200)
plt.show()

## I.6 El espectro

$L = D - A$ y $L_{\text{sym}} = I - D^{-1/2} A D^{-1/2}$, con `np.linalg.eigh`.
`networkx` solo para verificar.

In [ ]:
def laplacians(A, isolated_zero=True):
    # isolated_zero: convención para nodos de grado 0 (ver la explicación en (a))
    d = A.sum(axis=1)
    L = np.diag(d) - A
    d_inv_sqrt = np.zeros_like(d)
    d_inv_sqrt[d > 0] = 1 / np.sqrt(d[d > 0])
    L_sym = np.eye(len(d)) - d_inv_sqrt[:, None] * A * d_inv_sqrt[None, :]
    if isolated_zero:
        iso = d == 0
        L_sym[iso, iso] = 0
    return L, L_sym

L, L_sym = laplacians(A)
vals_L, vecs_L = np.linalg.eigh(L)
vals, vecs = np.linalg.eigh(L_sym)

seed_idx = ids.index(SEED_ID)
keep = [i for i in range(len(ids)) if i != seed_idx]
A_ns = A[np.ix_(keep, keep)]                 # sin la semilla
ids_ns = [ids[i] for i in keep]

### (a) Componentes

In [ ]:
def bfs_components(A):
    n = len(A)
    seen = np.zeros(n, dtype=bool)
    comps = []
    for s in range(n):
        if seen[s]:
            continue
        seen[s] = True
        comp, q = [s], deque([s])
        while q:
            u = q.popleft()
            for v in np.nonzero(A[u])[0]:
                if not seen[v]:
                    seen[v] = True
                    comp.append(v)
                    q.append(v)
        comps.append(comp)
    return comps

def zero_eigs(L_sym, tol=1e-8):
    return int(np.sum(np.linalg.eigvalsh(L_sym) < tol))

comps = bfs_components(A)
comps_ns = bfs_components(A_ns)
iso_ns = int(np.sum(A_ns.sum(axis=1) == 0))
_, L_sym_ns = laplacians(A_ns)
_, L_sym_ns_raw = laplacians(A_ns, isolated_zero=False)

print("Con la semilla:  BFS =", len(comps), "| valores propios < 1e-8 =", zero_eigs(L_sym))
print("Sin la semilla:  BFS =", len(comps_ns), f"(de ellas {iso_ns} actores aislados)")
print("  valores propios < 1e-8 poniendo solo D^-1/2 = 0:     ", zero_eigs(L_sym_ns_raw))
print("  valores propios < 1e-8 con la convención L_sym[i,i]=0:", zero_eigs(L_sym_ns))
print("Tamaños de las componentes sin la semilla:", sorted(map(len, comps_ns), reverse=True))
for comp in comps_ns:
    if len(comp) < 10:
        print("  componente chica:", [G.nodes[ids_ns[i]] for i in comp])

# En estos datos nadie queda aislado; un ejemplo chico muestra qué pasaría:
# 0 - 1 conectados y el nodo 2 aislado (2 componentes)
A_toy = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=float)
print("Ejemplo: valores propios solo con D^-1/2 = 0:", np.round(np.linalg.eigvalsh(laplacians(A_toy, False)[1]), 3))
print("Ejemplo: valores propios con L_sym[i,i] = 0: ", np.round(np.linalg.eigvalsh(laplacians(A_toy)[1]), 3))

# Verificación con networkx
nxG_ns = nx.Graph(); nxG_ns.add_nodes_from(range(len(A_ns)))
nxG_ns.add_edges_from(zip(*np.nonzero(np.triu(A_ns))))
print("networkx:", nx.number_connected_components(nxG), "y", nx.number_connected_components(nxG_ns))

**Convención para nodos aislados.** Si un nodo tiene grado 0 y se pone $D^{-1/2}_{ii}=0$, su fila y su columna
de $D^{-1/2}AD^{-1/2}$ quedan en cero, así que en $L_{\text{sym}}$ queda un 1 en la diagonal: ese nodo aporta el
valor propio **1**, no 0. Es una componente, pero no cuenta como tal, y el conteo sale corto (en el ejemplo chico hay
2 componentes y un solo cero). La convención que usamos (von Luxburg, 2007) es poner $L_{\text{sym},ii}=0$ para los
nodos de grado 0: así $e_i$ está en el núcleo y cada nodo aislado aporta su propio cero, igual que en $L=D-A$, donde
la fila de un nodo aislado ya es cero.

En el grafo de Harrison Ford nadie queda aislado al quitar la semilla, porque en cada película los otros miembros del
elenco principal quedan unidos entre sí. Sin Ford quedan **2 componentes**: una de 180 actores y otra de 4, el elenco de
*Cain Rose Up* (2017), un corto donde Ford sale solo con material de archivo como Han Solo. Lo único que unía a esos 4 con
el resto era Ford. Los filtros del enunciado no quitan el *archive footage*, y aquí se nota.

### (b) Brecha espectral

In [ ]:
lam2 = vals[1]
vals_ns = np.linalg.eigvalsh(L_sym_ns)
print(f"lambda_2 con la semilla: {lam2:.4f}")
print(f"lambda_2 sin la semilla: {vals_ns[1]:.2e}")

first = vals[:15]
gaps = np.diff(first)                       # gaps[i] = lambda_{i+2} - lambda_{i+1}
k1 = int(np.argmax(gaps)) + 1
print("Primeros 15 valores propios:", np.round(first, 4))
print(f"Brecha más grande: lambda_{k1+1} - lambda_{k1} = {gaps[k1-1]:.4f}  ->  k = {k1}")

# k = 1 no separa nada (es el efecto de la semilla, ver la respuesta); buscamos la mayor brecha con k >= 2
k = int(np.argmax(gaps[1:])) + 2
print(f"Mayor brecha con k >= 2: lambda_{k+1} - lambda_{k} = {gaps[k-1]:.4f}  ->  k = {k}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 16), first, "o-")
ax.axvspan(k1 + 0.05, k1 + 0.95, color="gray", alpha=0.2, label=f"brecha más grande (k = {k1})")
ax.axvspan(k + 0.05, k + 0.95, color="orange", alpha=0.3, label=f"mayor brecha con k ≥ 2 (k = {k})")
ax.set_xlabel("índice"); ax.set_ylabel(r"valor propio de $L_{sym}$")
ax.set_title("Primeros 15 valores propios (con la semilla)")
ax.legend(); plt.show()

# Verificación con networkx
nx_vals = np.sort(np.linalg.eigvalsh(nx.normalized_laplacian_matrix(nxG).toarray()))
print("Coincide con networkx:", np.allclose(nx_vals, vals))

**¿Por qué con la semilla el grafo es conexo y aun así la brecha delata comunidades?** Con la semilla
$\lambda_2 \approx 0.244$; sin ella $\lambda_2 \approx 0$ porque el grafo se parte en dos componentes. El grafo con la
semilla siempre es conexo por construcción: todos los nodos salieron de películas de Ford, así que todos son vecinos
de él (su grado es 184 = todos los demás). Por eso solo $\lambda_1 = 0$.

Pero un valor propio pequeño, aunque no sea cero, indica un **corte barato**: un grupo de nodos con pocas aristas hacia
afuera en relación con su volumen. Cada grupo de coprotagonistas se conecta con el resto casi solo a través de Ford, así
que hay varios cortes baratos y $\lambda_2,\dots,\lambda_5 \approx 0.24$–$0.30$ quedan juntos antes de un salto a
$\lambda_6 \approx 0.35$. La brecha más grande de todas es $\lambda_2-\lambda_1$ ($k=1$), que solo refleja que la semilla
lo une todo; por eso usamos la mayor brecha con $k \ge 2$: $\lambda_6-\lambda_5 = 0.054$, es decir **$k = 5$ comunidades**.

### (c) Clustering espectral

In [ ]:
U = vecs[:, :k]
U_norm = U / np.linalg.norm(U, axis=1, keepdims=True)
labels = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(U_norm)

for c in range(k):
    members = [ids[i] for i in np.nonzero(labels == c)[0]]
    top_actors = sorted(members, key=lambda n: -deg[n])[:3]
    movie_count = Counter(mid for mid, s in movie_nodes.items() for n in members if n in s)
    top_movies = [f"{movie_title[mid]} [{cnt}]" for mid, cnt in movie_count.most_common(3)]
    print(f"Comunidad {c}  ({len(members)} actores)")
    print("   actores:  ", ", ".join(f"{G.nodes[n]} ({deg[n]})" for n in top_actors))
    print("   películas:", "; ".join(top_movies))

**¿Las comunidades corresponden a algo real?** Sí, en su mayoría:

- **Star Wars** (11 actores): Mark Hamill, Billy Dee Williams, Carrie Fisher, con la trilogía original. Es una franquicia.
- **Indiana Jones y Blade Runner** (45 actores, incluye a Ford): Tommy Lee Jones, Rutger Hauer; las películas más
  repetidas son *Raiders of the Lost Ark*, *Temple of Doom* y *Blade Runner*, el Ford de los 80.
- **Inicios de su carrera, finales de los 70** (9 actores): *The Frisco Kid*, *The Possessed* y películas para TV.
  Corresponde a una época.
- **Cine bélico europeo** (11 actores): Franco Nero, Robert Shaw, *Force 10 from Navarone*; aquí también caen los 4 de
  *Cain Rose Up*, que solo cuelgan de Ford.
- **El resto** (109 actores): dramas y thrillers de Hollywood de 1988 en adelante (*Working Girl*, *Frantic*,
  *What Lies Beneath*), con Willem Dafoe, Christopher Plummer y Gary Oldman. Es la comunidad menos homogénea:
  agrupa todo lo que no forma un bloque cerrado.